# 階段一：資料品質檢查

本 Notebook 負責：
1. 下載並載入 2023–2025 年觀光旅館 XLSX 資料
2. 確認欄位結構與型態
3. 處理缺失值與異常值
4. 合併三年資料並輸出清理後 CSV

In [ ]:
# 套件安裝與基本設定
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "openpyxl", "requests", "matplotlib", "seaborn"], check=True)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Microsoft JhengHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'src'))

RAW_DIR = Path('..') / 'data' / 'raw'
PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('套件載入完成')
print(f'Pandas 版本：{pd.__version__}')

## 1. 下載觀光旅館營運統計資料

In [ ]:
from download_data import main as download_main
download_main()

## 2. 載入並初步檢查 XLSX 欄位結構

In [ ]:
# 分年載入，檢查各年欄位結構
frames_raw = {}
for year in [2023, 2024, 2025]:
    path = RAW_DIR / f'hotel_{year}.xlsx'
    if not path.exists():
        print(f'[{year}] 檔案不存在，請先執行下載')
        continue
    
    # 試讀不同 sheet
    xl = pd.ExcelFile(path, engine='openpyxl')
    print(f'\n[{year}] Sheet 清單：{xl.sheet_names}')
    
    df_raw = pd.read_excel(path, sheet_name=0, header=0, engine='openpyxl')
    frames_raw[year] = df_raw
    print(f'  圖表大小：{df_raw.shape}')
    print(f'  欄位清單：{list(df_raw.columns)}')

In [ ]:
# 顯示第一年資料前 10 筆
# 《重要》實際載入後再根據實際欄位名稱調整 preprocessing.py 中的 rename_map
if 2023 in frames_raw:
    df_sample = frames_raw[2023]
    display(df_sample.head(10))
    print('\n數據型態：')
    display(df_sample.dtypes)

## 3. 資料合併與清理

> **注意**：實際下載 XLSX 後，請先執行上方的欄位檢查區塊，確認實際欄位名稱後，再更新 `src/preprocessing.py` 中的 `rename_map`。

In [ ]:
from preprocessing import merge_all_years, save_processed

# 合併三年資料
df = merge_all_years()

print(f'\n合併後資料大小：{df.shape}')
print(f'欄位清單：{list(df.columns)}')

## 4. 資料品質檢查

In [ ]:
# 資料品質檢查
print('=== 基本資訊 ===')
print(f'資料筆數：{len(df)}')
if 'hotel_name' in df.columns:
    print(f'旅館數：{df["hotel_name"].nunique()}')
if 'year_month' in df.columns:
    print(f'月份範圍：{df["year_month"].min()} ~ {df["year_month"].max()}')

print('\n=== 缺失值統計 ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
if len(missing_df) > 0:
    display(missing_df)
else:
    print('無缺失值')

print('\n=== 住房率基本統計 ===')
if 'occupancy_rate' in df.columns:
    print(df['occupancy_rate'].describe())
    
print('\n=== 平均房價基本統計 ===')
if 'avg_price' in df.columns:
    print(df['avg_price'].describe())

## 5. 異常值檢查與處理

In [ ]:
# 異常值檢查
print('=== 異常住房率 ===')
if 'occupancy_rate' in df.columns:
    abnormal_occ = df[(df['occupancy_rate'] > 100) | (df['occupancy_rate'] < 0)]
    print(f'住房率超過100%或小於0%：{len(abnormal_occ)} 筆')
    if len(abnormal_occ) > 0:
        display(abnormal_occ.head(10))

print('\n=== 異常房價 ===')
if 'avg_price' in df.columns:
    abnormal_price = df[(df['avg_price'] <= 0) | (df['avg_price'] > 50000)]
    print(f'房價異常：{len(abnormal_price)} 筆')

print('\n=== 住房率分佈圖 ===')
if 'occupancy_rate' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 住房率分佈
    axes[0].hist(df['occupancy_rate'].dropna(), bins=30, color='#38bdf8', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('住房率 (%)')
    axes[0].set_ylabel('次數')
    axes[0].set_title('住房率分佈')
    axes[0].axvline(df['occupancy_rate'].mean(), color='red', linestyle='--', label=f'平均 {df["occupancy_rate"].mean():.1f}%')
    axes[0].legend()
    
    # 房價分佈
    if 'avg_price' in df.columns:
        axes[1].hist(df['avg_price'].dropna(), bins=30, color='#0ea5e9', edgecolor='white', alpha=0.8)
        axes[1].set_xlabel('平均房價 (NT$)')
        axes[1].set_ylabel('次數')
        axes[1].set_title('平均房價分佈')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/01_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 儲存清理後資料

In [ ]:
# 儲存清理後 CSV
save_processed(df)

print(f'\n清理後資料筆數：{len(df)}')
print(f'欄位數：{len(df.columns)}')
print(f'欄位清單：{list(df.columns)}')
print('\n✅ 資料品質檢查完成！')
print('下一步：執行 02_eda.ipynb 進行探索性資料分析')